In [2]:
# src/main_baseline.py

import os
import sys
import time
import logging
import pandas as pd
from pathlib import Path

# Fix OpenMP library conflict (common with PyTorch + other libraries)
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

sys.path.append(str(Path.cwd()))
from config.config import BaselineConfig
from data.data_loader import load_train_data
from pipelines.baseline_pipeline import BaselineRAGPipeline

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def main():
    timings = {}
    main_start = time.time()
    
    logger.info("Starting BASELINE RAG SYSTEM (System 1)")
    
    config = BaselineConfig()
    pipeline = BaselineRAGPipeline(config)
    
    # Phase 1: Index Building/Loading
    build_start = time.time()
    pipeline.build_index(force_rebuild=False)
    timings['Index Build/Load'] = time.time() - build_start

    # Phase 2: Component Initialization
    init_start = time.time()
    pipeline.initialize_components()
    timings['Component Initialization'] = time.time() - init_start

    # Phase 3: Data Loading and Filtering
    data_start = time.time()
    train_data = load_train_data(config.TRAIN_JSONL)
    pure_text_data = [r for r in train_data if r.get("types") == ["Pure-text (Plain-text)"]]
    timings['Data Loading/Filtering'] = time.time() - data_start

    logger.info(f"Loaded {len(train_data)} total questions, filtered to {len(pure_text_data)} pure-text questions")

    # Phase 4: Single Query Test
    sample_query = pure_text_data[0]
    query_start = time.time()
    result = pipeline.run_query(sample_query["question"])
    timings['Single Query Test'] = time.time() - query_start
    
    logger.info(f"Single query completed. Question: {sample_query['question']} | Ground Truth: {sample_query['answer']} | Generated: {result['answer']}")

    # Phase 5: Full Evaluation
    eval_start = time.time()
    metrics = pipeline.evaluate(pure_text_data[:20])
    eval_time = time.time() - eval_start
    timings['Full Evaluation (20 qs)'] = eval_time

    main_total = time.time() - main_start
    timings['Total Runtime'] = main_total
    
    print(f"[OK] Evaluation completed in {eval_time:.2f}s\n")
    
    # ===== SUMMARY REPORT =====
    print(f"\n{'='*80}")
    print("BASELINE SYSTEM - TIMING SUMMARY")
    print(f"{'='*80}\n")
    
    print(f"Phase Breakdown:")
    print(f"  1. Index Build/Load:               {timings['Index Build/Load']:>10.2f}s")
    print(f"  2. Component Initialization:       {timings['Component Initialization']:>10.2f}s")
    print(f"  3. Data Loading/Filtering:         {timings['Data Loading/Filtering']:>10.2f}s")
    print(f"  4. Single Query Test:              {timings['Single Query Test']:>10.2f}s")
    print(f"  5. Full Evaluation (20 qs):        {timings['Full Evaluation (20 qs)']:>10.2f}s")
    print(f"  " + "-" * 50)
    
    print(f"  TOTAL RUNTIME:                     {timings['Total Runtime']:>10.2f}s\n")
    
    # Print metrics if available
    if metrics:
        print(f"\n{'='*80}")
        print("EVALUATION METRICS")
        print(f"{'='*80}\n")
        
        if 'retrieval' in metrics:
            print("Retrieval Metrics:")
            for key, value in metrics['retrieval'].items():
                if isinstance(value, float):
                    print(f"  {key:<20}: {value:.4f}")
                else:
                    print(f"  {key:<20}: {value}")
        
        if 'generation' in metrics:
            print("\nGeneration Metrics:")
            for key, value in metrics['generation'].items():
                if isinstance(value, float):
                    print(f"  {key:<20}: {value:.4f}")
                else:
                    print(f"  {key:<20}: {value}")
        
        # Print phase-specific timing if available in metrics
        if 'timing' in metrics:
            print("\nPhase-Specific Timing (Evaluation):")
            print(f"  Retrieval Phase:   {metrics['timing'].get('phase1', 0):.2f}s")
            print(f"  Generation Phase:  {metrics['timing'].get('phase2', 0):.2f}s")
            print(f"  Total Evaluation:  {metrics['timing'].get('total', 0):.2f}s")
    
    print(f"\n{'='*80}\n")
    
    return metrics, timings

if __name__ == "__main__":
    mets, tims = main()
    

c:\Users\kiran\anaconda3\envs\dat560\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:__main__:Starting BASELINE RAG SYSTEM (System 1)
INFO:pipelines.base_pipeline:Building index with configuration: BaselineConfig...
INFO:pipelines.base_pipeline:Initializing embedder: jinaai/jina-clip-v2
INFO:indexing.embedder:Loading embedding model: jinaai/jina-clip-v2
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: jinaai/jina-clip-v2
C:\Users\kiran\.cache\huggingface\modules\transformers_modules\jinaai\jina-clip-implementation\39e6a55ae971b59bea6e44675d237c99762e7ee2\modeling_clip.py:140: UserWarning: Flash attention is not installed. Check https://github.com/Dao-AILab/flash-attention?tab=readme-ov-file#installation-and-features for installation instructions, disabling
  warnings.warn(
C


[OK] Embedding model loaded. Dimension: 1024
Initializing local Qdrant at c:\Users\kiran\Documents\_UIS\sem8\DAT560_GenerativAI_10\DAT560project\local_qdrant...


INFO:pipelines.base_pipeline:Collection check: 7702 documents found in Qdrant
INFO:pipelines.base_pipeline:Loading existing index with 7702 documents...
INFO:data.chunk_loader:Loading pre-processed chunks from c:\Users\kiran\Documents\_UIS\sem8\DAT560_GenerativAI_10\DAT560project\src\data\preprocessed\chunks_fixed_size.json
INFO:data.chunk_loader:Loaded 131 documents from JSON
INFO:data.chunk_loader:✓ Extracted 7702 chunks from 131 documents
INFO:data.chunk_loader:  Average chunks per document: 58.8
INFO:pipelines.base_pipeline:Index loaded in 31.07 seconds
INFO:pipelines.base_pipeline:Building hybrid BM25 + dense retriever...
INFO:indexing.hybrid_retriever:Building BM25 index over 7702 chunks...



[OK] Local Qdrant initialized


INFO:indexing.hybrid_retriever:BM25 index built.
INFO:pipelines.base_pipeline:Initializing generator with base URL https://ollama.ux.uis.no and model qwen3:32b
INFO:generation.generator:Loaded API key: sk-5585521...
INFO:generation.generator:Initialized Ollama client for model: qwen3:32b
INFO:__main__:Loaded 600 total questions, filtered to 85 pure-text questions
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.23s/it]
INFO:httpx:HTTP Request: POST https://ollama.ux.uis.no/api/chat "HTTP/1.1 200 OK"
INFO:__main__:Single query completed. Question: how many LEARNING OUTCOMES should be ANSWERed in UNIT 8?  | Ground Truth: 10 | Generated: 2
INFO:pipelines.baseline_pipeline:Evaluating on 20 test queries...

INFO:pipelines.baseline_pipeline:Processing query 1/20
Batches: 100%|██████████| 1/1 [00:00<00:00,  2.92it/s]
INFO:httpx:HTTP Request: POST https://ollama.ux.uis.no/api/chat "HTTP/1.1 200 OK"
INFO:pipelines.baseline_pipeline:Question: how many LEARNING OUTCOMES should be ANSWERed in UNIT 8


 === EVALUATION RESULTS ===
precision@1: 0.5500
precision@3: 0.2333
precision@5: 0.1400
recall@1: 0.5500
recall@3: 0.7000
recall@5: 0.7000
page_recall@1: 0.3000
page_recall@3: 0.4500
page_recall@5: 0.5500
ndcg@1: 0.5500
ndcg@3: 0.6381
ndcg@5: 0.6381
map: 0.6167
mrr: 0.6167
exact_match: 0.2500
contains_match: 0.3000
token_f1: 0.3953
bleu: 0.1999
rouge1: 0.4127
rouge2: 0.2477
rougeL: 0.4127
semantic_similarity: 0.3953

 === TIMING BREAKDOWN ===
Phase 1 (Query Processing): 69.07s
Phase 2 (Metric Computation): 0.03s
Total Evaluation Time: 69.10s
[OK] Evaluation completed in 69.10s


BASELINE SYSTEM - TIMING SUMMARY

Phase Breakdown:
  1. Index Build/Load:                    31.96s
  2. Component Initialization:             0.02s
  3. Data Loading/Filtering:               0.01s
  4. Single Query Test:                  813.34s
  5. Full Evaluation (20 qs):             69.10s
  --------------------------------------------------
  TOTAL RUNTIME:                         914.43s


EVALUATION ME

In [11]:
mets

{'precision@1': 0.55,
 'precision@3': 0.2333333333333333,
 'precision@5': 0.14000000000000004,
 'recall@1': 0.55,
 'recall@3': 0.7,
 'recall@5': 0.7,
 'page_recall@1': 0.3,
 'page_recall@3': 0.45,
 'page_recall@5': 0.55,
 'ndcg@1': 0.55,
 'ndcg@3': 0.6380929753571458,
 'ndcg@5': 0.6380929753571458,
 'map': 0.6166666666666667,
 'mrr': 0.6166666666666667,
 'exact_match': 0.25,
 'contains_match': 0.3,
 'token_f1': 0.39533315984815454,
 'bleu': 0.19987024754396554,
 'rouge1': 0.41274111331975777,
 'rouge2': 0.24773809523809526,
 'rougeL': 0.41274111331975777,
 'semantic_similarity': 0.39533315984815454,
 'timing': {'phase1': 69.06952238082886,
  'phase2': 0.03069472312927246,
  'total': 69.10116004943848}}